# Synthetic Data Augmentation with Conditional Flow Matching

## Part 3

In this section, we evaluate the impact of incorporating synthetic image data on classification performance. In particular, we examine changes in classification accuracy and score when the Fashion MNIST dataset is augmented with synthetic samples in the extremely low-data regime, using 0.2% of the available data for model training.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . > /dev/null 2>&1

In [2]:
!pip install -qU pip
!pip install -qU -r requirements.txt

In [3]:
import os
import sys
import argparse
import functools

# Silence tqdm output
os.environ["TQDM_DISABLE"] = "1"

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

We evaluate classification performance under synthetic data augmentation. 0.2% of the training set is combined with synthetic samples equivalent to 100% of the original dataset (6,000 per class), yielding a training set equal to 100.2% of the original size. Synthetic samples are generated with a guidance scale of $w = 3.0$, and performance is evaluated using both accuracy and the macro $\mathsf F_1$ score.

In [5]:
# Configure augmentation evaluation pipeline
augmentation_args = argparse.Namespace(data_root = "./data",
                                       batch_size = 16,
                                       num_workers = 0,
                                       epochs = 25,
                                       lr = 0.001,
                                       weight_decay = 1e-4,
                                       step_size = 15,
                                       gamma = 0.1,
                                       synthetic_data_dir = "./images",
                                       real_ratio = 0.002,
                                       classification_dir = "./results/classification",
                                       augmentation_dir = "./results/augmentation",
                                       checkpoint_dir = "./checkpoints",
                                       save_interval = 20,
                                       seed = 42)

# Create directory to store synthetic augmentation results
os.makedirs(augmentation_args.augmentation_dir, exist_ok = True)

# Set random seed for reproducibility
torch.manual_seed(augmentation_args.seed)
np.random.seed(augmentation_args.seed)

In [6]:
guidance_scale = 3.0
evaluator = SyntheticAugmentationEvaluator(augmentation_args, guidance_scale)
evaluator.run_low_data_experiments(augmentation_args.real_ratio, True)

Found 60000 synthetic images.
Number of epochs: 25
Number of training samples: 60120
Number of validation samples: 10000
Epoch 1/25
Training Set | Loss: 0.1609, Top-1 Accuracy: 95.69%, Top-5 Accuracy: 99.84%
Validation Set | Loss: 1.1224, Top-1 Accuracy: 74.99%, Top-5 Accuracy: 97.70%
Best Validation Loss (Up Until Now): 1.1224
_________________________________________________________________________________________________________

Epoch 2/25
Training Set | Loss: 0.0670, Top-1 Accuracy: 98.30%, Top-5 Accuracy: 99.97%
Validation Set | Loss: 1.2538, Top-1 Accuracy: 73.20%, Top-5 Accuracy: 96.20%
Best Validation Loss (Up Until Now): 1.1224
_________________________________________________________________________________________________________

Epoch 3/25
Training Set | Loss: 0.0523, Top-1 Accuracy: 98.70%, Top-5 Accuracy: 99.97%
Validation Set | Loss: 1.0823, Top-1 Accuracy: 76.24%, Top-5 Accuracy: 98.38%
Best Validation Loss (Up Until Now): 1.0823
______________________________________

Upon applying augmentation to the extremely low-data training split (i.e., 0.2% of the training set), the model achieves a classification accuracy of $79.56\%$, a macro $\mathsf F_1$ score of $0.7938$, and an optimal validation (cross-entropy) loss of $0.8625$ at epoch 9.